# Dataset Compilation

Merges annual CSV exports from `Dataset/Consolidated/` into clean compiled files in `Dataset/Compiled/`.

| Output file | Source pattern | Strategy |
|-------------|---------------|----------|
| `measurements.csv` | `measurements_*.csv` | concat → dedupe on Datetime |
| `greenhouse.csv` | `greenhouse_*.csv` | concat → dedupe on Datetime |
| `Animal_location_counts_*.csv` (×3) | `Animal_location_counts_* *.csv` | concat → dedupe on Date |
| `livestock_weight_long.csv` | `* Weight Data_Format 1_*.csv` (×3 species) | melt wide→long → dedupe |
| `livestock_condition_score_long.csv` | `* Condition Score Data_Format 1_*.csv` (×2 species) | melt wide→long → dedupe |
| `*_Location_Data_Format_1.csv` (×3) | `* Location Data_Format 1_*.csv` | concat wide → dedupe on Official tag |
| `*_Sales_Data_Format_1.csv` (×3) | `* Sales Data_Format 1_*.csv` | concat → dedupe on (tag, date) |
| `*_Basic_Data_Format_1.csv` (×3) | `* Basic Data_Format 1_*.csv` | concat → dedupe on Official tag |
| `Feed_Type_Data_Format_1.csv` | `Feed Type Data_Format 1_*.csv` | concat → dedupe |
| `Field_Event_Data_Format_1.csv` | `Field Event Data_Format 1_*.csv` | concat → dedupe |
| `Field_Survey_Data_Format_1_*.csv` (×5) | `Field Survey Data_Format 1_* *.csv` | concat per type → dedupe |

## 0. Imports & Config

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ── Resolve project root regardless of kernel working directory ─────────────
def _find_project_root():
    p = Path.cwd()
    if (p / 'Dataset/Consolidated').exists():
        return p
    for parent in p.parents:
        if (parent / 'Dataset/Consolidated').exists():
            return parent
    return p

PROJECT_ROOT = _find_project_root()
CONSOLIDATED = PROJECT_ROOT / 'Dataset/Consolidated'
COMPILED     = PROJECT_ROOT / 'Dataset/Compiled'
COMPILED.mkdir(exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Consolidated : {CONSOLIDATED}  (exists={CONSOLIDATED.exists()})')
print(f'Compiled     : {COMPILED}')

# Detect DD/MM/YYYY date columns (used in wide-format animal files)
DATE_PAT = re.compile(r'^\d{2}/\d{2}/\d{4}$')

def is_date_col(c):
    return bool(DATE_PAT.match(str(c)))

def load_and_concat(files, label='', **kwargs):
    """Load a list of CSV paths, concat, return DataFrame.
    Defaults to latin-1 encoding (raw exports contain µ, ® and other non-UTF-8 bytes).
    Pass encoding='utf-8' to override for known UTF-8 files."""
    files = sorted(files)
    if not files:
        print(f'  WARNING: no files found for {label}')
        return pd.DataFrame()
    kwargs.setdefault('encoding', 'latin-1')
    frames = [pd.read_csv(f, low_memory=False, **kwargs) for f in files]
    df = pd.concat(frames, ignore_index=True)
    print(f'  {label}: {len(files)} files → {len(df):,} rows × {df.shape[1]} cols')
    return df

def melt_wide_to_long(df, value_name, id_cols=None):
    """
    Melt wide-format animal data (rows=animals, date cols) to long format.
    Returns DataFrame with id_cols + Record_Date + value_name.
    """
    if id_cols is None:
        id_cols = [c for c in df.columns if not is_date_col(c)]
    date_cols = [c for c in df.columns if is_date_col(c)]
    melted = df.melt(
        id_vars=id_cols, value_vars=date_cols,
        var_name='Record_Date', value_name=value_name
    )
    melted[value_name] = pd.to_numeric(
        melted[value_name].replace('NA', np.nan), errors='coerce'
    )
    melted = melted.dropna(subset=[value_name])
    melted['Record_Date'] = pd.to_datetime(
        melted['Record_Date'], format='%d/%m/%Y', errors='coerce'
    )
    melted = melted.dropna(subset=['Record_Date'])
    return melted.reset_index(drop=True)

print('Helpers loaded.')

Project root : c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project
Consolidated : c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\Dataset\Consolidated  (exists=True)
Compiled     : c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\Dataset\Compiled
Helpers loaded.


## 1. Sensor Time-Series
### 1a. Measurements (15-min)

In [21]:
files_m = sorted(CONSOLIDATED.glob('measurements_*.csv'))
df_meas = load_and_concat(files_m, 'measurements')

df_meas['Datetime'] = pd.to_datetime(df_meas['Datetime'],
                                     format='%Y/%m/%d %H:%M:%S', errors='coerce')
before = len(df_meas)
df_meas = df_meas.sort_values('Datetime').drop_duplicates(subset='Datetime').reset_index(drop=True)
print(f'  Duplicates removed: {before - len(df_meas)}')
print(f'  Period: {df_meas["Datetime"].min().date()} → {df_meas["Datetime"].max().date()}')
print(f'  Final shape: {df_meas.shape}')

out = COMPILED / 'measurements.csv'
df_meas.to_csv(out, index=False)
print(f'  Saved → {out.name}')

  measurements: 8 files → 281,280 rows × 718 cols
  Duplicates removed: 672
  Period: 2017-01-01 → 2025-01-02
  Final shape: (280608, 718)
  Saved → measurements.csv


### 1b. Greenhouse GHG (30-min)

In [22]:
files_g = sorted(CONSOLIDATED.glob('greenhouse_*.csv'))
df_ghg = load_and_concat(files_g, 'greenhouse')

df_ghg['Datetime'] = pd.to_datetime(df_ghg['Datetime'],
                                    format='%Y/%m/%d %H:%M:%S', errors='coerce')
before = len(df_ghg)
df_ghg = df_ghg.sort_values('Datetime').drop_duplicates(subset='Datetime').reset_index(drop=True)
print(f'  Duplicates removed: {before - len(df_ghg)}')
print(f'  Period: {df_ghg["Datetime"].min().date()} → {df_ghg["Datetime"].max().date()}')
print(f'  Final shape: {df_ghg.shape}')

out = COMPILED / 'greenhouse.csv'
df_ghg.to_csv(out, index=False, encoding='utf-8')
print(f'  Saved → {out.name}')

  greenhouse: 7 files → 122,972 rows × 295 cols
  Duplicates removed: 293
  Period: 2017-01-01 → 2024-01-01
  Final shape: (122679, 295)
  Saved → greenhouse.csv


## 2. Animal Location Counts (Daily)

In [23]:
LOC_COUNT_GROUPS = {
    'Breeding_Sheep_Basic_Data': 'Animal_location_counts_Breeding Sheep Basic Data_*.csv',
    'Cattle_Basic_Data':          'Animal_location_counts_Cattle Basic Data_*.csv',
    'Lamb_Basic_Data':            'Animal_location_counts_Lamb Basic Data_*.csv',
}

for out_stem, pattern in LOC_COUNT_GROUPS.items():
    files = sorted(CONSOLIDATED.glob(pattern))
    df = load_and_concat(files, out_stem)
    if df.empty:
        continue
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    before = len(df)
    df = df.sort_values('Date').drop_duplicates(subset='Date').reset_index(drop=True)
    print(f'  Duplicates removed: {before - len(df)}')
    print(f'  Period: {df["Date"].min().date()} → {df["Date"].max().date()}')
    out = COMPILED / f'Animal_location_counts_{out_stem}.csv'
    df.to_csv(out, index=False)
    print(f'  Saved → {out.name}\n')

  Breeding_Sheep_Basic_Data: 8 files → 2,938 rows × 22 cols
  Duplicates removed: 14
  Period: 2017-01-01 → 2025-01-01
  Saved → Animal_location_counts_Breeding_Sheep_Basic_Data.csv

  Cattle_Basic_Data: 8 files → 2,938 rows × 22 cols
  Duplicates removed: 14
  Period: 2017-01-01 → 2025-01-01
  Saved → Animal_location_counts_Cattle_Basic_Data.csv

  Lamb_Basic_Data: 8 files → 2,938 rows × 22 cols
  Duplicates removed: 14
  Period: 2017-01-01 → 2025-01-01
  Saved → Animal_location_counts_Lamb_Basic_Data.csv



## 3. Livestock Weight & Condition Score (Wide → Long)
### 3a. Weight

In [24]:
WEIGHT_SPECIES = {
    'Breeding Sheep': 'Breeding Sheep Weight Data_Format 1_*.csv',
    'Cattle':         'Cattle Weight Data_Format 1_*.csv',
    'Lamb':           'Lamb Weight Data_Format 1_*.csv',
}

weight_parts = []
for species, pattern in WEIGHT_SPECIES.items():
    files = sorted(CONSOLIDATED.glob(pattern))
    raw = load_and_concat(files, f'weight/{species}')
    if raw.empty:
        continue
    # id cols: Official tag + Category (both present in weight files)
    id_cols = [c for c in ['Official tag', 'Category'] if c in raw.columns]
    long = melt_wide_to_long(raw, value_name='Weight_kg', id_cols=id_cols)
    long['Species'] = species
    print(f'  {species}: {len(long):,} non-null observations after melt')
    weight_parts.append(long)

wt_all = pd.concat(weight_parts, ignore_index=True)
# Drop duplicate (animal, date) pairs — same record in multiple annual files
before = len(wt_all)
wt_all = wt_all.drop_duplicates(subset=['Official tag', 'Record_Date'], keep='last').reset_index(drop=True)
print(f'\nDuplicates removed: {before - len(wt_all)}')
print(f'Final shape: {wt_all.shape}')

out = COMPILED / 'livestock_weight_long.csv'
wt_all.to_csv(out, index=False)
print(f'Saved → {out.name}')

  weight/Breeding Sheep: 8 files → 1,799 rows × 399 cols
  Breeding Sheep: 45,938 non-null observations after melt
  weight/Cattle: 8 files → 1,587 rows × 311 cols
  Cattle: 21,813 non-null observations after melt
  weight/Lamb: 8 files → 2,459 rows × 202 cols
  Lamb: 22,276 non-null observations after melt

Duplicates removed: 41885
Final shape: (48142, 5)
Saved → livestock_weight_long.csv


### 3b. Condition Score

In [25]:
# Note: Lamb has no condition score files
CS_SPECIES = {
    'Breeding Sheep': 'Breeding Sheep Condition Score Data_Format 1_*.csv',
    'Cattle':         'Cattle Condition Score Data_Format 1_*.csv',
}

cs_parts = []
for species, pattern in CS_SPECIES.items():
    files = sorted(CONSOLIDATED.glob(pattern))
    raw = load_and_concat(files, f'condition/{species}')
    if raw.empty:
        continue
    # Condition score files have only 'Official tag' (no Category column)
    id_cols = [c for c in ['Official tag', 'Category'] if c in raw.columns]
    long = melt_wide_to_long(raw, value_name='Score', id_cols=id_cols)
    long['Species'] = species
    print(f'  {species}: {len(long):,} non-null observations after melt')
    cs_parts.append(long)

cs_all = pd.concat(cs_parts, ignore_index=True)
before = len(cs_all)
cs_all = cs_all.drop_duplicates(subset=['Official tag', 'Record_Date'], keep='last').reset_index(drop=True)
print(f'\nDuplicates removed: {before - len(cs_all)}')
print(f'Final shape: {cs_all.shape}')

out = COMPILED / 'livestock_condition_score_long.csv'
cs_all.to_csv(out, index=False)
print(f'Saved → {out.name}')

  condition/Breeding Sheep: 8 files → 1,785 rows × 385 cols
  Breeding Sheep: 43,369 non-null observations after melt
  condition/Cattle: 4 files → 552 rows × 41 cols
  Cattle: 3,540 non-null observations after melt

Duplicates removed: 30128
Final shape: (16781, 4)
Saved → livestock_condition_score_long.csv


## 4. Livestock Location Data (Wide Format — Preserved)

Rows = individual animals, columns = dates with location values. Kept wide for downstream use.

In [26]:
LOCATION_SPECIES = {
    'Breeding_Sheep': 'Breeding Sheep Location Data_Format 1_*.csv',
    'Cattle':         'Cattle Location Data_Format 1_*.csv',
    'Lamb':           'Lamb Location Data_Format 1_*.csv',
}

for stem, pattern in LOCATION_SPECIES.items():
    files = sorted(CONSOLIDATED.glob(pattern))
    df = load_and_concat(files, f'location/{stem}')
    if df.empty:
        continue
    before = len(df)
    # Each annual file contains the full history for all animals in that cohort.
    # Keep one row per animal (the last file's version has the most complete columns).
    df = df.drop_duplicates(subset='Official tag', keep='last').reset_index(drop=True)
    print(f'  Duplicates removed: {before - len(df)}  |  unique animals: {len(df):,}')
    date_cols = [c for c in df.columns if is_date_col(c)]
    print(f'  Date columns: {len(date_cols)} ({date_cols[0]} → {date_cols[-1]})')
    out = COMPILED / f'{stem}_Location_Data_Format_1.csv'
    df.to_csv(out, index=False)
    print(f'  Saved → {out.name}\n')

  location/Breeding_Sheep: 8 files → 1,814 rows × 850 cols
  Duplicates removed: 1049  |  unique animals: 765
  Date columns: 849 (01/01/2012 → 31/12/2025)
  Saved → Breeding_Sheep_Location_Data_Format_1.csv

  location/Cattle: 8 files → 1,587 rows × 368 cols
  Duplicates removed: 729  |  unique animals: 858
  Date columns: 367 (02/11/2015 → 21/04/2026)
  Saved → Cattle_Location_Data_Format_1.csv

  location/Lamb: 8 files → 2,459 rows × 614 cols
  Duplicates removed: 118  |  unique animals: 2,341
  Date columns: 613 (23/03/2017 → 07/11/2024)
  Saved → Lamb_Location_Data_Format_1.csv



## 5. Sales Data

In [27]:
SALES_SPECIES = {
    'Breeding_Sheep': 'Breeding Sheep Sales Data_Format 1_*.csv',
    'Cattle':         'Cattle Sales Data_Format 1_*.csv',
    'Lamb':           'Lamb Sales Data_Format 1_*.csv',
}

for stem, pattern in SALES_SPECIES.items():
    files = sorted(CONSOLIDATED.glob(pattern))
    df = load_and_concat(files, f'sales/{stem}')
    if df.empty:
        continue
    df['Date sold'] = pd.to_datetime(df['Date sold'], dayfirst=True, errors='coerce')
    before = len(df)
    df = df.drop_duplicates(subset=['Official tag', 'Date sold'], keep='last').reset_index(drop=True)
    print(f'  Duplicates removed: {before - len(df)}  |  unique sales: {len(df):,}')
    out = COMPILED / f'{stem}_Sales_Data_Format_1.csv'
    df.to_csv(out, index=False)
    print(f'  Saved → {out.name}\n')

  sales/Breeding_Sheep: 8 files → 1,611 rows × 11 cols
  Duplicates removed: 1037  |  unique sales: 574
  Saved → Breeding_Sheep_Sales_Data_Format_1.csv

  sales/Cattle: 8 files → 1,587 rows × 11 cols
  Duplicates removed: 729  |  unique sales: 858
  Saved → Cattle_Sales_Data_Format_1.csv

  sales/Lamb: 8 files → 2,459 rows × 11 cols
  Duplicates removed: 118  |  unique sales: 2,341
  Saved → Lamb_Sales_Data_Format_1.csv



## 6. Basic Data (Animal Pedigree / Identity)

In [28]:
BASIC_SPECIES = {
    'Breeding_Sheep': 'Breeding Sheep Basic Data_Format 1_*.csv',
    'Cattle':         'Cattle Basic Data_Format 1_*.csv',
    'Lamb':           'Lamb Basic Data_Format 1_*.csv',
}

for stem, pattern in BASIC_SPECIES.items():
    files = sorted(CONSOLIDATED.glob(pattern))
    df = load_and_concat(files, f'basic/{stem}')
    if df.empty:
        continue
    before = len(df)
    df = df.drop_duplicates(subset='Official tag', keep='last').reset_index(drop=True)
    print(f'  Duplicates removed: {before - len(df)}  |  unique animals: {len(df):,}')
    out = COMPILED / f'{stem}_Basic_Data_Format_1.csv'
    df.to_csv(out, index=False)
    print(f'  Saved → {out.name}\n')

  basic/Breeding_Sheep: 8 files → 1,814 rows × 11 cols
  Duplicates removed: 1049  |  unique animals: 765
  Saved → Breeding_Sheep_Basic_Data_Format_1.csv

  basic/Cattle: 8 files → 1,587 rows × 12 cols
  Duplicates removed: 729  |  unique animals: 858
  Saved → Cattle_Basic_Data_Format_1.csv

  basic/Lamb: 8 files → 2,459 rows × 14 cols
  Duplicates removed: 118  |  unique animals: 2,341
  Saved → Lamb_Basic_Data_Format_1.csv



## 7. Feed Type & Field Events

In [29]:
# Feed Type
files_feed = sorted(CONSOLIDATED.glob('Feed Type Data_Format 1_*.csv'))
df_feed = load_and_concat(files_feed, 'Feed Type')
df_feed['Sampling_Date'] = pd.to_datetime(df_feed['Sampling_Date'], dayfirst=True, errors='coerce')
before = len(df_feed)
df_feed = df_feed.drop_duplicates(keep='last').reset_index(drop=True)
print(f'  Duplicates removed: {before - len(df_feed)}  |  final: {len(df_feed):,} rows')
out = COMPILED / 'Feed_Type_Data_Format_1.csv'
df_feed.to_csv(out, index=False)
print(f'  Saved → {out.name}')

print()

# Field Events
files_ev = sorted(CONSOLIDATED.glob('Field Event Data_Format 1_*.csv'))
df_ev = load_and_concat(files_ev, 'Field Event')
df_ev['Event_Date'] = pd.to_datetime(df_ev['Event_Date'], dayfirst=True, errors='coerce')
before = len(df_ev)
df_ev = df_ev.drop_duplicates(keep='last').reset_index(drop=True)
print(f'  Duplicates removed: {before - len(df_ev)}  |  final: {len(df_ev):,} rows')
out = COMPILED / 'Field_Event_Data_Format_1.csv'
df_ev.to_csv(out, index=False)
print(f'  Saved → {out.name}')

  Feed Type: 8 files → 11,896 rows × 23 cols
  Duplicates removed: 10409  |  final: 1,487 rows
  Saved → Feed_Type_Data_Format_1.csv

  Field Event: 8 files → 2,148 rows × 19 cols
  Duplicates removed: 1  |  final: 2,147 rows
  Saved → Field_Event_Data_Format_1.csv


## 8. Field Surveys (5 types)

In [30]:
SURVEY_TYPES = {
    'Botanical_Survey':         'Field Survey Data_Format 1_Botanical Survey_*.csv',
    'Grain_Survey':             'Field Survey Data_Format 1_Grain Survey_*.csv',
    'Herbage_Survey':           'Field Survey Data_Format 1_Herbage Survey_*.csv',
    'Silage_Cut_Survey':        'Field Survey Data_Format 1_Silage Cut Survey_*.csv',
    'Soil_Chemistry__Physics':  'Field Survey Data_Format 1_Soil Chemistry & Physics_*.csv',
}

for out_stem, pattern in SURVEY_TYPES.items():
    files = sorted(CONSOLIDATED.glob(pattern))
    df = load_and_concat(files, out_stem)
    if df.empty:
        continue
    # Detect date column
    date_col = next((c for c in ['Sample_date', 'Sampling_Date', 'Date'] if c in df.columns), None)
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    before = len(df)
    df = df.drop_duplicates(keep='last').reset_index(drop=True)
    print(f'  Duplicates removed: {before - len(df)}  |  final: {len(df):,} rows')
    out = COMPILED / f'Field_Survey_Data_Format_1_{out_stem}.csv'
    df.to_csv(out, index=False)
    print(f'  Saved → {out.name}\n')

  Botanical_Survey: 2 files → 657 rows × 165 cols
  Duplicates removed: 0  |  final: 657 rows
  Saved → Field_Survey_Data_Format_1_Botanical_Survey.csv

  Grain_Survey: 4 files → 204 rows × 81 cols
  Duplicates removed: 0  |  final: 204 rows
  Saved → Field_Survey_Data_Format_1_Grain_Survey.csv

  Herbage_Survey: 7 files → 1,620 rows × 69 cols
  Duplicates removed: 166  |  final: 1,454 rows
  Saved → Field_Survey_Data_Format_1_Herbage_Survey.csv

  Silage_Cut_Survey: 3 files → 245 rows × 15 cols
  Duplicates removed: 0  |  final: 245 rows
  Saved → Field_Survey_Data_Format_1_Silage_Cut_Survey.csv

  Soil_Chemistry__Physics: 7 files → 1,739 rows × 114 cols
  Duplicates removed: 166  |  final: 1,573 rows
  Saved → Field_Survey_Data_Format_1_Soil_Chemistry__Physics.csv



## 9. Verification — Check All Outputs

In [31]:
import os

expected = [
    'measurements.csv',
    'greenhouse.csv',
    'Animal_location_counts_Breeding_Sheep_Basic_Data.csv',
    'Animal_location_counts_Cattle_Basic_Data.csv',
    'Animal_location_counts_Lamb_Basic_Data.csv',
    'livestock_weight_long.csv',
    'livestock_condition_score_long.csv',
    'Breeding_Sheep_Location_Data_Format_1.csv',
    'Cattle_Location_Data_Format_1.csv',
    'Lamb_Location_Data_Format_1.csv',
    'Breeding_Sheep_Sales_Data_Format_1.csv',
    'Cattle_Sales_Data_Format_1.csv',
    'Lamb_Sales_Data_Format_1.csv',
    'Breeding_Sheep_Basic_Data_Format_1.csv',
    'Cattle_Basic_Data_Format_1.csv',
    'Lamb_Basic_Data_Format_1.csv',
    'Feed_Type_Data_Format_1.csv',
    'Field_Event_Data_Format_1.csv',
    'Field_Survey_Data_Format_1_Botanical_Survey.csv',
    'Field_Survey_Data_Format_1_Grain_Survey.csv',
    'Field_Survey_Data_Format_1_Herbage_Survey.csv',
    'Field_Survey_Data_Format_1_Silage_Cut_Survey.csv',
    'Field_Survey_Data_Format_1_Soil_Chemistry__Physics.csv',
]

rows = []
for fname in expected:
    fpath = COMPILED / fname
    if fpath.exists():
        df = pd.read_csv(fpath, low_memory=False, nrows=0)
        size_mb = round(fpath.stat().st_size / 1e6, 1)
        # Count rows quickly
        with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
            nrows = sum(1 for _ in f) - 1
        rows.append({'file': fname, 'rows': nrows, 'cols': df.shape[1], 'size_MB': size_mb, 'status': 'OK'})
    else:
        rows.append({'file': fname, 'rows': None, 'cols': None, 'size_MB': None, 'status': 'MISSING'})

summary = pd.DataFrame(rows)
print(f'Compiled files: {(summary["status"]=="OK").sum()} / {len(summary)}')
display(summary)

Compiled files: 23 / 23


,file,rows,cols,size_MB,status
0,measurements.csv,280608,718,1526.9,OK
1,greenhouse.csv,122679,295,144.0,OK
2,Animal_location_counts_Breeding_Sheep_Basic_Da...,2924,22,0.3,OK
3,Animal_location_counts_Cattle_Basic_Data.csv,2924,22,0.3,OK
4,Animal_location_counts_Lamb_Basic_Data.csv,2924,22,0.3,OK
5,livestock_weight_long.csv,48142,5,2.4,OK
6,livestock_condition_score_long.csv,16781,4,0.8,OK
7,Breeding_Sheep_Location_Data_Format_1.csv,765,850,0.9,OK
8,Cattle_Location_Data_Format_1.csv,858,368,0.4,OK
9,Lamb_Location_Data_Format_1.csv,2341,614,1.6,OK
